# A database of national culture scores from 987 published studies, 1968–2020 Exploration with `mlcroissant`
This notebook provides a guided template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.r3pn-6whh/fair2.json`

This dataset describes tabular data extracted from published studies reporting national culture measures, organized into linked tables for studies, samples, scores, within-sample correlations, and attrition. In this notebook, all dataset entities are referenced using their `@id` values whenever available.

In [ ]:
# Ensure required libraries are installed
!pip install -q mlcroissant pandas matplotlib seaborn

## 1. Data Loading
Load metadata from the dataset using `mlcroissant`. We keep the Croissant URL in a variable so the notebook can be reused easily with another schema if needed.

In [ ]:
import json
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.r3pn-6whh/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata_json = dataset.metadata.to_json()

print('Dataset name:', getattr(dataset.metadata, 'name', 'Unknown'))
print('Dataset description:', getattr(dataset.metadata, 'description', 'No description available'))
print('Croissant version:', metadata_json.get('conformsTo'))
print('Dataset @id:', metadata_json.get('@id'))
print('Identifier:', metadata_json.get('identifier'))
print('Version:', metadata_json.get('version'))

We can also inspect a few high-level metadata fields from the JSON representation. This is useful for understanding the scope, provenance, and intended use of the dataset.

In [ ]:
summary_fields = [
    'name', 'description', 'license', 'citeAs', 'temporalCoverage',
    'spatialCoverage', 'keywords', 'datePublished'
]

for field_id in summary_fields:
    print(f'\n{field_id}:')
    value = metadata_json.get(field_id)
    print(value)

## 2. Data Overview
Review the available record sets, fields, and their IDs.

Some Croissant packages expose record sets directly in metadata. This dataset metadata snapshot indicates that `recordSet` may be empty at the top level, so the code below is written defensively: it first checks what is available and then attempts to inspect the dataset object itself.

In [ ]:
# Inspect recordSet entries from the metadata JSON
record_sets_meta = metadata_json.get('recordSet', []) or []
print('Number of record sets declared in metadata_json:', len(record_sets_meta))

for rs in record_sets_meta:
    rs_id = rs.get('@id', 'N/A') if isinstance(rs, dict) else 'N/A'
    rs_name = rs.get('name', 'N/A') if isinstance(rs, dict) else 'N/A'
    print(f'Record set @id: {rs_id} | name: {rs_name}')

In [ ]:
# Inspect attributes available on dataset.metadata and dataset for exploration
print('Attributes on dataset.metadata:')
print([attr for attr in dir(dataset.metadata) if not attr.startswith('_')][:100])

print('\nAttributes on dataset:')
print([attr for attr in dir(dataset) if not attr.startswith('_')][:100])

If record sets are available through the Croissant loader, the next helper cell extracts record set and field information using their `@id` values. If they are not exposed, the notebook still remains useful for metadata exploration and can be adapted once record set IDs are known.

In [ ]:
# Build a compact overview of available record sets and fields if present
record_set_ids = []
record_set_overview = []

for rs in record_sets_meta:
    if not isinstance(rs, dict):
        continue
    rs_id = rs.get('@id')
    rs_name = rs.get('name')
    fields = rs.get('field', []) or []
    field_ids = []
    for field in fields:
        if isinstance(field, dict):
            field_ids.append(field.get('@id'))
    record_set_ids.append(rs_id)
    record_set_overview.append({
        'record_set_@id': rs_id,
        'record_set_name': rs_name,
        'field_count': len(field_ids),
        'field_@ids': field_ids
    })

overview_df = pd.DataFrame(record_set_overview)
overview_df

The next cell demonstrates how to inspect records from a specific record set by `@id`. It uses a variable rather than a hard-coded string so you can update it once a valid record set `@id` is identified.

In [ ]:
# Replace this variable with a valid record set @id if available from the metadata or documentation
example_record_set_id = record_set_ids[0] if record_set_ids else None
print('Example record set @id:', example_record_set_id)

if example_record_set_id is not None:
    try:
        for i, record in enumerate(dataset.records(record_set=example_record_set_id)):
            print(record)
            if i >= 2:
                break
    except Exception as e:
        print('Unable to iterate over records for this record set @id.')
        print(type(e).__name__, str(e))
else:
    print('No record set @id was discovered in metadata_json. Continue with metadata exploration or provide known record set @ids manually.')

## 3. Data Extraction
Load data from one or more specific record sets into pandas DataFrames for analysis. As required, the notebook is designed to use record set `@id` values.

If no record sets are exposed directly by the current metadata, you can manually populate `record_sets_to_load` once the valid `@id` values are available.

In [ ]:
# Extract data from each record set using its @id
record_sets_to_load = record_set_ids.copy()
dataframes = {}

for record_set_id in record_sets_to_load:
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f'Loaded {len(dataframes[record_set_id])} rows from record set @id: {record_set_id}')
    except Exception as e:
        print(f'Could not load record set @id: {record_set_id}')
        print(type(e).__name__, str(e))

print('\nLoaded DataFrame keys:')
print(list(dataframes.keys()))

In [ ]:
# Show columns and preview for the first successfully loaded record set
if dataframes:
    first_record_set_id = next(iter(dataframes))
    print('First loaded record set @id:', first_record_set_id)
    print('Columns:')
    print(dataframes[first_record_set_id].columns.tolist())
    display(dataframes[first_record_set_id].head())
else:
    print('No record sets were loaded. If you know the record set @ids for studies, samples, scores, correlations, or attrition tables, assign them to record_sets_to_load and rerun this section.')

To support reusable analysis, the following helper function scans loaded DataFrames and summarizes candidate numeric columns. This is especially helpful when field `@id` values are not yet documented locally.

In [ ]:
def summarize_loaded_dataframes(frames):
    rows = []
    for rs_id, df in frames.items():
        numeric_cols = df.select_dtypes(include='number').columns.tolist()
        rows.append({
            'record_set_@id': rs_id,
            'n_rows': len(df),
            'n_columns': len(df.columns),
            'numeric_columns': numeric_cols
        })
    return pd.DataFrame(rows)

summarize_loaded_dataframes(dataframes)

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering records, normalizing numeric fields, checking missingness, and grouping by a categorical field.

Because Croissant datasets can vary in structure, this section is written to automatically choose a loaded record set and a numeric field where possible. If you know the exact field `@id` values, you should replace the automatically selected fields with those IDs.

In [ ]:
if dataframes:
    analysis_record_set_id = next(iter(dataframes))
    analysis_df = dataframes[analysis_record_set_id].copy()
    print('Analysis record set @id:', analysis_record_set_id)
    print('Shape:', analysis_df.shape)
else:
    analysis_record_set_id = None
    analysis_df = pd.DataFrame()
    print('No DataFrame available for EDA.')

In [ ]:
# Identify candidate numeric and grouping fields
if not analysis_df.empty:
    numeric_fields = analysis_df.select_dtypes(include='number').columns.tolist()
    non_numeric_fields = analysis_df.select_dtypes(exclude='number').columns.tolist()

    numeric_field_id = numeric_fields[0] if numeric_fields else None
    group_field_id = non_numeric_fields[0] if non_numeric_fields else None

    print('Candidate numeric field @id/name:', numeric_field_id)
    print('Candidate group field @id/name:', group_field_id)
else:
    numeric_field_id = None
    group_field_id = None

In [ ]:
# Filtering, normalization, and grouping example
if numeric_field_id is not None:
    numeric_series = pd.to_numeric(analysis_df[numeric_field_id], errors='coerce')
    valid_numeric = numeric_series.dropna()

    if not valid_numeric.empty:
        threshold = valid_numeric.quantile(0.50)
        filtered_df = analysis_df[numeric_series > threshold].copy()
        filtered_numeric = pd.to_numeric(filtered_df[numeric_field_id], errors='coerce')

        print(f'Filtered records with {numeric_field_id} > {threshold}:')
        display(filtered_df.head())

        if filtered_numeric.std() not in [0, None] and pd.notnull(filtered_numeric.std()):
            filtered_df[f'{numeric_field_id}_normalized'] = (
                filtered_numeric - filtered_numeric.mean()
            ) / filtered_numeric.std()
            print(f'Normalized {numeric_field_id} for filtered records:')
            display(filtered_df[[numeric_field_id, f'{numeric_field_id}_normalized']].head())
        else:
            print('Standard deviation is zero or undefined; normalization skipped.')

        if group_field_id is not None and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id, dropna=False)[numeric_field_id].mean().sort_values(ascending=False)
            print(f'Grouped mean of {numeric_field_id} by {group_field_id}:')
            display(grouped_df.head(10).to_frame(name=f'mean_{numeric_field_id}'))
    else:
        print('No valid numeric values available for filtering.')
else:
    print('No numeric field available for EDA in the loaded DataFrame.')

In [ ]:
# Missing data overview
if not analysis_df.empty:
    missing_summary = analysis_df.isna().mean().sort_values(ascending=False).to_frame('missing_fraction')
    print('Top columns by missing fraction:')
    display(missing_summary.head(15))
else:
    print('No DataFrame available for missing data analysis.')

For culture-score datasets, outlier checks are often useful. The next cell demonstrates a simple IQR-based outlier flag for the chosen numeric field.

In [ ]:
if numeric_field_id is not None:
    numeric_series = pd.to_numeric(analysis_df[numeric_field_id], errors='coerce')
    valid_numeric = numeric_series.dropna()

    if len(valid_numeric) >= 4:
        q1 = valid_numeric.quantile(0.25)
        q3 = valid_numeric.quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr

        analysis_df[f'{numeric_field_id}_outlier_flag'] = (
            (numeric_series < lower_bound) | (numeric_series > upper_bound)
        )

        print('Outlier detection summary:')
        print('Lower bound:', lower_bound)
        print('Upper bound:', upper_bound)
        print('Number of flagged rows:', analysis_df[f'{numeric_field_id}_outlier_flag'].sum())
        display(analysis_df[[numeric_field_id, f'{numeric_field_id}_outlier_flag']].head())
    else:
        print('Not enough numeric data points for IQR-based outlier detection.')
else:
    print('No numeric field available for outlier analysis.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

The plotting code is also defensive: it only runs when an appropriate numeric field is available.

In [ ]:
if numeric_field_id is not None:
    plot_series = pd.to_numeric(analysis_df[numeric_field_id], errors='coerce').dropna()
    if not plot_series.empty:
        plt.figure(figsize=(10, 5))
        sns.histplot(plot_series, kde=True, bins=30)
        plt.title(f'Distribution of {numeric_field_id}')
        plt.xlabel(numeric_field_id)
        plt.ylabel('Count')
        plt.show()
    else:
        print('No valid numeric values available for histogram.')
else:
    print('No numeric field available for visualization.')

In [ ]:
if numeric_field_id is not None and group_field_id is not None:
    temp_df = analysis_df[[group_field_id, numeric_field_id]].copy()
    temp_df[numeric_field_id] = pd.to_numeric(temp_df[numeric_field_id], errors='coerce')
    temp_df = temp_df.dropna(subset=[numeric_field_id])

    top_groups = temp_df[group_field_id].astype(str).value_counts().head(10).index
    plot_df = temp_df[temp_df[group_field_id].astype(str).isin(top_groups)].copy()

    if not plot_df.empty:
        plt.figure(figsize=(12, 6))
        sns.boxplot(data=plot_df, x=group_field_id, y=numeric_field_id)
        plt.title(f'{numeric_field_id} by {group_field_id} (top 10 groups)')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()
    else:
        print('Insufficient grouped data for boxplot visualization.')
else:
    print('A numeric field and a grouping field are both required for grouped visualization.')

## 6. Conclusion
This notebook demonstrated how to:

- load a Croissant dataset with `mlcroissant`,
- inspect dataset-level metadata,
- discover record sets and fields using their `@id` values when available,
- extract record sets into pandas DataFrames,
- perform basic filtering, normalization, grouping, missing-data checks, and outlier detection,
- and create simple visualizations for numeric fields.

For this FAIR^2 package, the metadata indicates a rich resource on national culture scores derived from published studies spanning 1968–2020. The dataset is especially suitable for meta-research, coverage analysis, and structured study/sample/scale-level synthesis. If record set `@id` values are available from the live Croissant endpoint or supplementary documentation, you can plug them directly into the extraction cells above to analyze specific tables such as studies, samples, scores, correlations, or attrition.